In [6]:
import pandas as pd

# Load the dataset into the 'unimproved' DataFrame
unimproved = pd.read_csv('../dataset/unimproved_subset.csv')

# Display the first few rows of the DataFrame
unimproved.head()

C:\Users\manager\AppData\Local\Temp\ipykernel_12884\3820460364.py:4: DtypeWarning: Columns (16: COUNTY, 38: MAIN_HEATING_CONTROLS, 52: FLOOR_ENV_EFF) have mixed types. Specify dtype option on import or set low_memory=False.
  unimproved = pd.read_csv('../dataset/unimproved_subset.csv')


,column00,LMK_KEY,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,BUILDING_REFERENCE_NUMBER,CURRENT_ENERGY_RATING,POTENTIAL_ENERGY_RATING,CURRENT_ENERGY_EFFICIENCY,...,POTENTIAL_GAIN,wall_score,roof_score,window_score,CONSTRUCTION_AGE_GROUP,retrofit_cost_score,easy_win_index,cluster,LAT,LON
0,5746222,6486699ff420afd4b7185e5e39a38c9dacea5bbd29e790...,19 Malt Mill Close,Kilsby,NaN,CV23 8XN,10001397893,F,D,36,...,26,3,3,2,1976-1982,8,208,0,52.350955,-1.30389
1,5746915,a39ed1b6881c17e5bad49a83e048ffefa220e75f503633...,5 Manor Road,Kilsby,NaN,CV23 8XS,10001524950,E,C,43,...,34,1,1,2,<1900,4,136,1,52.350955,-1.30389
2,5750373,ebdc67ce65aef64713faf4f6ffbccf835f1685aadcf880...,End Cottage,Chapel Street,Kilsby,CV23 8XE,2078675278,E,C,47,...,29,1,3,2,<1900,6,174,0,52.350955,-1.30389
3,5752105,955c14effbc1b4fe505d0dcb02b34dc2c05a6c4369e6cc...,8 The Green,Lilbourne,NaN,CV23 0SR,10002433585,F,D,31,...,35,1,3,2,<1900,6,210,0,52.350955,-1.30389
4,5754808,8e9a5a312e5b6922afb93e807b4affdc2736a7e75ade6e...,23 Hall Close,Kilsby,NaN,CV23 8XZ,10002619382,E,D,41,...,27,3,3,2,1967-1975,8,216,0,52.350955,-1.30389


In [7]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
import joblib

# -----------------------------
# 1. Select Features + Targets
# -----------------------------
features = [
    'CURRENT_ENERGY_RATING',
    'CURRENT_ENERGY_EFFICIENCY',
    'BUILT_FORM',
    'ENERGY_CONSUMPTION_CURRENT',
    'CO2_EMISSIONS_CURRENT',
    'HEATING_COST_CURRENT',
    'HOT_WATER_COST_CURRENT',
    'TOTAL_FLOOR_AREA',
    'GLAZED_TYPE',
    'WINDOWS_DESCRIPTION',
    'WALLS_DESCRIPTION',
    'WALLS_ENERGY_EFF',
    'ROOF_DESCRIPTION',
    'ROOF_ENERGY_EFF',
    'MAINHEAT_DESCRIPTION',
    'MAINHEAT_ENERGY_EFF',
    'CONSTRUCTION_AGE_GROUP'
]

# Fixed: Removed categorical POTENTIAL_ENERGY_RATING target for regression
targets = [
    'POTENTIAL_ENERGY_EFFICIENCY'
]

X = unimproved[features]
y = unimproved[targets].values.ravel()  # Flatten to a 1D array for single-output regression

# -----------------------------
# 2. Split Data
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 3. Identify Column Types
# -----------------------------
text_features = [
    'CURRENT_ENERGY_RATING',
    'BUILT_FORM',
    'GLAZED_TYPE',
    'WINDOWS_DESCRIPTION',
    'WALLS_DESCRIPTION',
    'WALLS_ENERGY_EFF',
    'ROOF_DESCRIPTION',
    'ROOF_ENERGY_EFF',
    'MAINHEAT_DESCRIPTION',
    'MAINHEAT_ENERGY_EFF',
    'CONSTRUCTION_AGE_GROUP'
]

numeric_features = [
    'CURRENT_ENERGY_EFFICIENCY',
    'ENERGY_CONSUMPTION_CURRENT',
    'CO2_EMISSIONS_CURRENT',
    'HEATING_COST_CURRENT',
    'HOT_WATER_COST_CURRENT',
    'TOTAL_FLOOR_AREA'
]

# -----------------------------
# 4. Column Transformer
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ('text', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), text_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# -----------------------------
# 5. Boosting Model (Single-Output Regressor)
# -----------------------------
# Fixed: MultiOutputRegressor is no longer required since we have a single numeric target
boost_model = HistGradientBoostingRegressor(max_depth=8, learning_rate=0.1)

pipeline = make_pipeline(preprocess, boost_model)

# -----------------------------
# 6. Train Model
# -----------------------------
pipeline.fit(X_train, y_train)

# -----------------------------
# 7. Save Model
# -----------------------------
joblib.dump(pipeline, 'epc_boost_model.pkl')

print("Model saved as epc_boost_model.pkl")

Model saved as epc_boost_model.pkl


In [8]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
import joblib

# 1. Load the trained model
model = joblib.load("epc_boost_model.pkl")

# 2. Create an example input row (must match your feature names)
example_house = pd.DataFrame([{
    'CURRENT_ENERGY_RATING': 'D',
    'BUILT_FORM': 'Semi-Detached',
    'GLAZED_TYPE': 'Double glazing installed',
    'WINDOWS_DESCRIPTION': 'Mostly double glazed',
    'WALLS_DESCRIPTION': 'Cavity wall, as built, insulated',
    'WALLS_ENERGY_EFF': 'Good',
    'ROOF_DESCRIPTION': 'Pitched roof',
    'ROOF_ENERGY_EFF': 'Average',
    'MAINHEAT_DESCRIPTION': 'Gas boiler, standard efficiency',
    'MAINHEAT_ENERGY_EFF': 'Poor',
    'CONSTRUCTION_AGE_GROUP': '1990-2000',

    'CURRENT_ENERGY_EFFICIENCY': 62,
    'ENERGY_CONSUMPTION_CURRENT': 250,
    'CO2_EMISSIONS_CURRENT': 3.5,
    'HEATING_COST_CURRENT': 650,
    'HOT_WATER_COST_CURRENT': 120,
    'TOTAL_FLOOR_AREA': 85
}])

# 3. Predict potential EPC efficiency score
prediction = model.predict(example_house)

# 4. Extract result
potential_efficiency = prediction[0]

print("Predicted POTENTIAL_ENERGY_EFFICIENCY:", potential_efficiency)

def efficiency_to_rating(score):
    if score >= 92:
        return "A"
    elif score >= 81:
        return "B"
    elif score >= 69:
        return "C"
    elif score >= 55:
        return "D"
    elif score >= 39:
        return "E"
    elif score >= 21:
        return "F"
    else:
        return "G"


 
 
rating = efficiency_to_rating(potential_efficiency)
print(rating)   # B



Predicted POTENTIAL_ENERGY_EFFICIENCY: 72.56820017805191
C
